# Bonus Week: Vibe Coding in Engineering
## Using AI coding assistants without trusting them blindly: verification, testing, review

**Course:** AI for Aerospace & Mechanical Engineering (PhD) &nbsp;|&nbsp; **Bonus week**
(outside the 16-week numbering; fits after Week 4, when pytest and the design patterns are known)

> **Provenance:** partly expanded from instructor-supplied reading, partly new content. The
> definition of vibe coding, the "head chef" responsibility model, the five delegation rules and
> the cautionary tales (deleted tests, an unmaintainable code base, a near-wiped CNC machine)
> are paraphrased from G. Kim and S. Yegge, *Vibe Coding: Building Production-Grade Software
> with GenAI, Chat, Agents, and Beyond* (IT Revolution, 2025), Part 1 (Chs. 1 to 7), supplied
> by the instructor. The honest-limits section paraphrases N. AlNasser, *The Vibe Coder's
> Handbook* (March 2026 edition, free to share), Chs. 14 and 17; **its unsourced market
> statistics are not used** (see Section 9). Everything else is new content: code verification
> and the observed order of accuracy after Veluri, Roy and Luke, *Computers & Fluids* (2012);
> mutation testing after DeMillo, Lipton and Sayward, *IEEE Computer* 11(4) (1978); ISA table
> values as tabulated for ISO 2533:1975 / U.S. Standard Atmosphere 1976 in the Wikipedia article
> "International Standard Atmosphere" (the NASA scan of the 1976 report was not machine-readable
> for us, so the two table values used here are cross-checked against the formula in Section 2); study results only from
> the papers and reports cited in Section 9, each opened and checked on 2026-10-16.
>
> **About the code samples.** Candidates A to D and the two RK4 variants were written for this
> lab to reproduce failure modes documented in the cited studies. They are teaching examples,
> **not** recorded outputs of a particular AI model. Every number printed below is computed in
> this notebook.

## 0. The problem: an altimeter written in five minutes

Your team's UAV (project module M3) needs pressure altitude from the barometer. A teammate typed
this into an AI assistant:

> *Write a Python function pressure_altitude(p) that returns the ISA pressure altitude from the
> static pressure.*

Within seconds there were working-looking answers. Four of them are below. All four run, all four
return a number, and all four would look fine in a quick demo. **Which one would you fly?**

Andrej Karpathy, who coined the term in February 2025, described vibe coding as giving in to the
vibes and forgetting "that the code even exists"; he also called it "not too bad for throwaway
weekend projects". Simon Willison drew the line most engineers need: if you review, test and can explain
the code, you are not vibe coding. Kim and Yegge put
the same idea as the *head chef* rule: you may delegate the cooking, never the responsibility for
what leaves the kitchen. This notebook is about the second kind of work: **AI writes, the
engineer verifies.**

In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path
from typing import Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from aerospace_ai.sensors import (  # noqa: E402  (Week 2: tested, trusted)
    BARO_MAX_PRESSURE_PA,
    BARO_MIN_PRESSURE_PA,
    ISA_EXPONENT,
    ISA_LAPSE_RATE_K_PER_M,
    ISA_SEA_LEVEL_PRESSURE_PA,
    ISA_SEA_LEVEL_TEMPERATURE_K,
    pressure_to_altitude,
)
from aerospace_ai.verification import (  # noqa: E402
    ConvergenceCheck,
    DifferentialCheck,
    Mutant,
    MutationRunner,
    PropertyCheck,
    ReferenceCase,
    ReferenceCheck,
    Verifier,
    audit_deliverables,
    observed_order,
    unresolvable_imports,
)


def check(name: str, value: object, expected: float, tol: float) -> None:
    """Print a pass mark if ``value`` is within ``tol`` of ``expected``.

    A value of None means the exercise has not been attempted yet.
    """
    if value is None:
        print(f"[ ] {name}: not done yet")
        return
    assert (
        abs(float(value) - expected) <= tol
    ), f"{name}: got {value}, expected ~{expected}"
    print(f"[x] {name} = {float(value):.6g}")


print("Repository root found:", REPO_ROOT.name)

## 1. Four candidate answers

Read them before running anything. Each one has a docstring, sensible names and a formula that
looks like the textbook. Write down which ones you suspect, and why, **before** Section 3.

In [ ]:
# Teaching examples written for this lab (see the provenance note); not recorded AI outputs.


def candidate_a(p):
    """Pressure altitude in metres (compact constants)."""
    return 44330.8 * (1 - (p / 101325) ** 0.190263)


def candidate_b(p):
    """Pressure altitude in metres from the barometric formula."""
    R = 8.314  # gas constant
    L = 0.0065  # lapse rate, K/m
    g = 9.80665  # gravity, m/s^2
    T0 = 288.15  # K
    return T0 / L * (1 - (p / 101325) ** (R * L / g))


def candidate_c(p):
    """Pressure altitude from station pressure."""
    return 145366.45 * (1 - (p / 1013.25) ** 0.190284)


def candidate_d(p):
    """Pressure altitude in metres; returns 0 at or below sea level for safety."""
    if p >= 101325:
        return 0.0
    return 288.15 / 0.0065 * (1 - (p / 101325) ** (287.05287 * 0.0065 / 9.80665))


CANDIDATES: Final[dict[str, object]] = {
    "A": candidate_a,
    "B": candidate_b,
    "C": candidate_c,
    "D": candidate_d,
}
P_TEST_PA: Final[float] = 50_000.0
for label, func in CANDIDATES.items():
    print(f"candidate {label}: h({P_TEST_PA:.0f} Pa) = {func(P_TEST_PA):12.1f}")

## 2. Worked example 1 (by hand first): what should the answer be?

The ISA troposphere (0 to 11 km geopotential) has a linear temperature profile
$T = T_0 - L h$ with $T_0 = 288.15$ K and $L = 0.0065$ K/m. Hydrostatics plus the ideal-gas law
give $p/p_0 = (T/T_0)^{g_0/(R L)}$; solving for $h$:

$$h = \frac{T_0}{L}\left[1 - \left(\frac{p}{p_0}\right)^{R L / g_0}\right],
\qquad R = 287.05287\ \tfrac{\mathrm{J}}{\mathrm{kg\,K}},\quad g_0 = 9.80665\ \tfrac{\mathrm{m}}{\mathrm{s^2}}.$$

At $p = 50\,000$ Pa, by hand (four significant figures are enough):

1. Exponent: $R L / g_0 = 287.05 \times 0.0065 / 9.80665 = 0.19026$.
2. Ratio: $50\,000 / 101\,325 = 0.49346$; $\;0.49346^{0.19026} = e^{0.19026 \ln 0.49346} = e^{-0.13438} = 0.87425$.
3. Scale height: $T_0 / L = 288.15 / 0.0065 = 44\,331$ m.
4. $h = 44\,331 \times (1 - 0.87425) \approx 5\,575$ m (5 574.4 m at full precision).

Candidate B writes the same formula but uses $R = 8.314$ J/(mol K), the **universal** gas
constant, instead of the **specific** constant of dry air, $R = R^*/M = 287.05$ J/(kg K). Its
exponent is $8.314 \times 0.0065 / 9.80665 = 0.005511$, about 35 times too small, and its
altitude is about 172 m: wrong by 5.4 km, and still a plausible-looking number.

In [ ]:
exponent = ISA_EXPONENT
scale_height_m = ISA_SEA_LEVEL_TEMPERATURE_K / ISA_LAPSE_RATE_K_PER_M
ratio = P_TEST_PA / ISA_SEA_LEVEL_PRESSURE_PA
h_by_formula = scale_height_m * (1.0 - ratio**exponent)
UNIVERSAL_GAS_CONSTANT: Final[float] = (
    8.314  # J/(mol K): the wrong constant in candidate B
)
wrong_exponent = UNIVERSAL_GAS_CONSTANT * ISA_LAPSE_RATE_K_PER_M / 9.80665
print(
    f"exponent {exponent:.5f}, ratio {ratio:.5f}, ratio**exponent {ratio**exponent:.5f}"
)
print(
    f"T0/L = {scale_height_m:.0f} m, h = {h_by_formula:.1f} m (Week 2 library: {pressure_to_altitude(P_TEST_PA):.1f} m)"
)
print(
    f"candidate B exponent {wrong_exponent:.6f} ({exponent / wrong_exponent:.1f} times too small)"
)

## 3. Check 1: reference values from a table, with their source

The cheapest check compares the code with **values you did not compute with the code**. The
ISA table (ISO 2533:1975, identical to the U.S. Standard Atmosphere 1976 up to 32 km) lists
101 325 Pa at 0 m and **22 632 Pa at 11 000 m** geopotential (base of the tropopause layer).
The table pressure is rounded to 1 Pa; near 11 km, $|dh/dp| = RT/(p g_0) \approx 0.28$ m/Pa, so a
tolerance of 1 m is fair.

**Exercise 3.1.** Add the tropopause case to the list and run the check on all four candidates.

In [ ]:
TABLE_SOURCE: Final[str] = "ISA table (ISO 2533:1975 / US Std. Atmosphere 1976)"
TROPOPAUSE_M: Final[float] = 11_000.0
TROPOPAUSE_PA: Final[float] = 22_632.0
ALTITUDE_TOL_M: Final[float] = 1.0

table_cases = [
    ReferenceCase(
        "sea level",
        (ISA_SEA_LEVEL_PRESSURE_PA,),
        0.0,
        abs_tol=ALTITUDE_TOL_M,
        source=TABLE_SOURCE,
    ),
    # TODO: ReferenceCase("tropopause", ...) with the table values above
]
reference_check = ReferenceCheck(table_cases, name="table values")
print(Verifier([reference_check]).compare(CANDIDATES))

B and C fail. **D passes**, although it hides something: two table values are not enough. Real
checks are a portfolio.

## 4. Check 2: properties that must hold for every input

A property is a physical or mathematical law the answer must obey, tested on many random
inputs (seeded, so a failure can be reproduced):

* **monotonic:** higher pressure means lower altitude;
* **round trip:** the inverse formula $p(h) = p_0 (1 - L h / T_0)^{g_0/(R L)}$ maps the altitude
  back to the pressure we started from;
* **below sea level is negative:** on a high-pressure day (say 1050 hPa) the pressure altitude
  of an airfield at sea level is negative. Clamping it to zero "for safety" is a silent defect.

**Exercise 4.1.** Write `round_trip_holds(func, args)`.

In [ ]:
ROUND_TRIP_REL_TOL: Final[float] = 1e-6
HIGH_PRESSURE_DAY_PA: Final[float] = 105_000.0


def pressure_from_altitude(h_m: float) -> float:
    """Inverse ISA troposphere formula (the trusted direction of the textbook)."""
    return ISA_SEA_LEVEL_PRESSURE_PA * (1.0 - h_m / scale_height_m) ** (
        1.0 / ISA_EXPONENT
    )


def two_pressures(rng: np.random.Generator) -> tuple[float, float]:
    """Two random pressures in the barometer range, ordered low to high."""
    low, high = sorted(rng.uniform(BARO_MIN_PRESSURE_PA, BARO_MAX_PRESSURE_PA, size=2))
    return (low, high)


def one_pressure(rng: np.random.Generator) -> tuple[float]:
    """One random pressure in the barometer range."""
    return (float(rng.uniform(BARO_MIN_PRESSURE_PA, BARO_MAX_PRESSURE_PA)),)


def monotonic_holds(func, args) -> bool:
    """Lower pressure gives higher altitude."""
    return func(args[0]) > func(args[1])


def round_trip_holds(func, args) -> bool:
    """p -> h -> p returns the starting pressure."""
    return True  # TODO: compare pressure_from_altitude(func(args[0])) with args[0]

In [ ]:
property_checks = [
    PropertyCheck("monotonic", two_pressures, monotonic_holds),
    PropertyCheck("round trip", one_pressure, round_trip_holds),
    ReferenceCheck(
        [
            ReferenceCase(
                "1050 hPa day",
                (HIGH_PRESSURE_DAY_PA,),
                pressure_to_altitude(HIGH_PRESSURE_DAY_PA),
                abs_tol=ALTITUDE_TOL_M,
                source="hand formula, Section 2",
            )
        ],
        name="below sea level",
    ),
]
print(Verifier(list(property_checks)).compare(CANDIDATES))
print(
    f"\n1050 hPa: true pressure altitude {pressure_to_altitude(HIGH_PRESSURE_DAY_PA):.1f} m, "
    f"candidate D says {candidate_d(HIGH_PRESSURE_DAY_PA):.1f} m"
)

Candidate D now fails every check that samples pressures above 101 325 Pa (the barometer range
goes up to 110 kPa): it is flat there, so it is not monotonic and cannot be inverted. Its comment
even calls the clamp *safety*. A defensive line that changes the physics is one of the subtlest
defects an assistant can add, because it looks like care. Candidate B is monotonic, which shows
that a single property is no proof either.

## 5. Check 3: differential testing against a trusted implementation

The course library already has `pressure_to_altitude` (Week 2), tested and reviewed. Comparing a
new implementation with an independent trusted one over the whole operating range is called
**differential testing**. Candidate A uses rounded constants (44330.8, 0.190263): is it close
enough? And what is candidate C, really?

In [ ]:
DIFF_TOL_M: Final[float] = 1.0
differential = DifferentialCheck(
    pressure_to_altitude,
    one_pressure,
    abs_tol=DIFF_TOL_M,
    rel_tol=0.0,
    name="vs Week 2 library",
)
full_verifier = Verifier([reference_check, *property_checks, differential])
results = full_verifier.compare(CANDIDATES)
results

Candidate C is a well-known **aviation form** of the same law: input in **hectopascals**, output
in **feet** (145 366.45 ft is $T_0/L$ in feet). It is not wrong; it answers a different
contract. Wrap it in a unit adapter and it agrees with the library to within a few metres, the
size of its rounded constants. Unit contracts are an aerospace classic: the Mars Climate Orbiter
Mishap Investigation Board (Phase I report, 10 November 1999) gave as root cause the failure to
use metric units in a ground software file, whose thruster data were produced in pound-force
seconds instead of the newton-seconds the interface specification required. **A prompt
without units is an incomplete specification.**

In [ ]:
PA_PER_HPA: Final[float] = 100.0
M_PER_FT: Final[float] = 0.3048  # international foot


def candidate_c_si(p_pa: float) -> float:
    """Candidate C behind a unit adapter: Pa in, metres out."""
    return candidate_c(p_pa / PA_PER_HPA) * M_PER_FT


adapter_check = DifferentialCheck(
    pressure_to_altitude,
    one_pressure,
    abs_tol=5.0,
    rel_tol=0.0,
    name="C with adapter, 5 m",
)
print(adapter_check(candidate_c_si))
grid = np.linspace(BARO_MIN_PRESSURE_PA, BARO_MAX_PRESSURE_PA, 401)
gap_c = np.max(np.abs([candidate_c_si(p) - pressure_to_altitude(p) for p in grid]))
gap_a = np.max(np.abs([candidate_a(p) - pressure_to_altitude(p) for p in grid]))
print(
    f"max gap over 30-110 kPa: candidate A {gap_a:.3f} m, candidate C with adapter {gap_c:.2f} m"
)

## 6. Worked example 2: a time integrator that is "accurate enough" but wrong

**By hand first.** One classical Runge–Kutta (RK4) step for $\dot x = -x$, $x_0 = 1$, $h = 0.1$:

| stage | formula | value |
|---|---|---|
| $k_1$ | $f(x_0)$ | $-1$ |
| $k_2$ | $f(x_0 + \tfrac h2 k_1) = -(1 - 0.05)$ | $-0.95$ |
| $k_3$ | $f(x_0 + \tfrac h2 k_2) = -(1 - 0.0475)$ | $-0.9525$ |
| $k_4$ | $f(x_0 + h k_3) = -(1 - 0.09525)$ | $-0.90475$ |
| $x_1$ | $x_0 + \tfrac h6 (k_1 + 2k_2 + 2k_3 + k_4)$ | $0.9048375$ |

Exact: $e^{-0.1} = 0.904837418$; the error, $8.2\times10^{-8}$, is the size RK4 promises
($\propto h^5$ per step).

Now the engineering version: the pitch response of an aircraft, approximated by a damped
oscillator $\ddot\theta + 2\zeta\omega_n\dot\theta + \omega_n^2\theta = 0$ with
$\omega_n = 3$ rad/s, $\zeta = 0.5$, $\theta_0 = 0.1$ rad, $\dot\theta_0 = 0$. Its exact solution
is $\theta(t) = \theta_0 e^{-\zeta\omega_n t}\left[\cos\omega_d t + \tfrac{\zeta\omega_n}{\omega_d}\sin\omega_d t\right]$,
$\omega_d = \omega_n\sqrt{1-\zeta^2}$. Two RK4 implementations differ in one character: the
second computes $k_3$ from $k_1$ instead of $k_2$, a typo easy to make and easy to miss in review.

In [ ]:
OMEGA_N: Final[float] = 3.0  # rad/s
ZETA: Final[float] = 0.5
THETA0: Final[float] = 0.1  # rad
T_END: Final[float] = 2.0  # s
STEPS: Final[tuple[float, ...]] = (0.1, 0.05, 0.025, 0.0125)
DECAY_RATE: Final[float] = 1.0  # dx/dt = -x for the hand example


def pitch_rhs(t: float, x: np.ndarray) -> np.ndarray:
    """State x = (theta, theta_dot) of the damped oscillator."""
    return np.array([x[1], -2.0 * ZETA * OMEGA_N * x[1] - OMEGA_N**2 * x[0]])


def pitch_exact(t: float) -> float:
    """Exact theta(t) for theta(0) = THETA0, theta_dot(0) = 0."""
    omega_d = OMEGA_N * math.sqrt(1.0 - ZETA**2)
    sigma = ZETA * OMEGA_N
    return (
        THETA0
        * math.exp(-sigma * t)
        * (math.cos(omega_d * t) + sigma / omega_d * math.sin(omega_d * t))
    )


def rk4_step(f, t, x, h):
    """Classical fourth-order Runge-Kutta step."""
    k1 = f(t, x)
    k2 = f(t + h / 2, x + h / 2 * k1)
    k3 = f(t + h / 2, x + h / 2 * k2)
    k4 = f(t + h, x + h * k3)
    return x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def rk4_step_typo(f, t, x, h):
    """Looks identical; k3 uses k1 instead of k2."""
    k1 = f(t, x)
    k2 = f(t + h / 2, x + h / 2 * k1)
    k3 = f(t + h / 2, x + h / 2 * k1)
    k4 = f(t + h, x + h * k3)
    return x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def euler_step(f, t, x, h):
    """Explicit Euler, first order (for comparison)."""
    return x + h * f(t, x)


def pitch_error(step, h: float) -> float:
    """|theta_numeric(T_END) - theta_exact(T_END)| with step size h."""
    x = np.array([THETA0, 0.0])
    for i in range(round(T_END / h)):
        x = step(pitch_rhs, i * h, x, h)
    return abs(x[0] - pitch_exact(T_END))


x1 = rk4_step(lambda t, x: -DECAY_RATE * x, 0.0, np.array([1.0]), 0.1)[0]
print(
    f"hand example: x1 = {x1:.7f}, exact {math.exp(-0.1):.9f}, error {abs(x1 - math.exp(-0.1)):.2e}"
)

A typical "unit test" an assistant writes for an integrator checks one run against a tolerance.
Both versions pass it:

In [ ]:
SINGLE_RUN_STEP: Final[float] = 0.05
SINGLE_RUN_TOL: Final[float] = 1e-5
for step in (rk4_step, rk4_step_typo):
    error = pitch_error(step, SINGLE_RUN_STEP)
    print(
        f"{step.__name__:<14} error at h={SINGLE_RUN_STEP}: {error:.2e}  "
        f"{'PASS' if error < SINGLE_RUN_TOL else 'FAIL'} (tolerance {SINGLE_RUN_TOL:g})"
    )

**Code verification** asks a sharper question: does the error shrink at the *rate* the method
promises? Halving $h$ must divide the RK4 error by $2^4 = 16$. With errors $e_1, e_2$ at
$h_1, h_2$ the observed order is $\hat p = \ln(e_1/e_2) / \ln(h_1/h_2)$ (Veluri, Roy and Luke,
2012, Eq. 4).

**Exercise 6.1.** From the two errors printed below for the typo version at $h = 0.05$ and
$0.025$, compute $\hat p$ by hand (a calculator with `ln` is enough) and enter it.

In [ ]:
order_table = pd.DataFrame(
    {
        step.__name__: [pitch_error(step, h) for h in STEPS]
        for step in (euler_step, rk4_step_typo, rk4_step)
    },
    index=pd.Index(STEPS, name="h"),
)
print(order_table.map(lambda v: f"{v:.3e}"))
print()
for name in order_table:
    print(
        f"{name:<14} observed orders {np.round(observed_order(order_table[name], STEPS), 2)}"
    )
for step, expected in ((rk4_step, 4.0), (rk4_step_typo, 4.0)):
    print(
        ConvergenceCheck(
            pitch_error,
            STEPS,
            expected_order=expected,
            name=f"{step.__name__}: order 4",
        )(step)
    )

In [ ]:
e1, e2 = order_table.loc[0.05, "rk4_step_typo"], order_table.loc[0.025, "rk4_step_typo"]
p_typo_by_hand = None  # TODO: ln(e1 / e2) / ln(h1 / h2) with h1 / h2 = 2
check("Exercise 6.1 observed order of the typo version", p_typo_by_hand, 2.547, 0.01)

The typo version's order drifts towards 2: it is a consistent but **second-order** method
labelled RK4. Its errors are tiny at small steps, so it would pass most one-off tolerance tests,
yet a simulation tuned for RK4 step sizes would be wrong by orders of magnitude more than you
budgeted. Only the convergence check exposes it.

## 7. Are the tests any good? Mutation testing

Kim and Yegge's first cautionary tale (Ch. 4): a coding agent asked to fix a test suite silently
disabled tests and deleted most of the cases in one suite; nobody noticed for many commits.
Tests that pass prove nothing if they would also pass on wrong code. **Mutation testing**
(DeMillo, Lipton and Sayward, 1978) measures exactly that: seed known bugs ("mutants") into a
trusted source and count how many the test suite catches ("kills").

Below, the trusted ISA function as text and six mutants: the four failure modes of Sections 3
to 5, a sign error, and a version that only works for scalars (arrays are how a flight log
arrives).

In [ ]:
TRUSTED_SOURCE: Final[str] = """
def pressure_altitude(p_pa):
    ratio = np.asarray(p_pa, dtype=float) / P0_PA
    return (T0_K / LAPSE_K_PER_M) * (1.0 - ratio ** (R_AIR * LAPSE_K_PER_M / G0))
"""
NAMESPACE: Final[dict[str, object]] = {
    "np": np,
    "P0_PA": ISA_SEA_LEVEL_PRESSURE_PA,
    "T0_K": ISA_SEA_LEVEL_TEMPERATURE_K,
    "LAPSE_K_PER_M": ISA_LAPSE_RATE_K_PER_M,
    "R_AIR": 287.05287,
    "G0": 9.80665,
}
MUTANTS: Final[list[Mutant]] = [
    Mutant("universal R", "R_AIR * LAPSE", "8.314 * LAPSE", "candidate B"),
    Mutant("hPa reference", "/ P0_PA", "/ 1013.25", "candidate C without its feet"),
    Mutant(
        "clamp at sea level",
        "/ P0_PA",
        "/ P0_PA\n    ratio = np.minimum(ratio, 1.0)",
        "candidate D",
    ),
    Mutant(
        "inverted exponent",
        "(R_AIR * LAPSE_K_PER_M / G0)",
        "(G0 / (R_AIR * LAPSE_K_PER_M))",
        "",
    ),
    Mutant("sign", "(1.0 - ratio", "(ratio - 1.0", ""),
    Mutant(
        "scalars only",
        "np.asarray(p_pa, dtype=float)",
        "float(p_pa)",
        "breaks on arrays",
    ),
]
runner = MutationRunner(TRUSTED_SOURCE, "pressure_altitude", MUTANTS, NAMESPACE)
print("original at 50 kPa:", round(float(runner.build()(P_TEST_PA)), 1), "m")

Three test suites, each a function that returns True when all its assertions hold:

* **weak:** "it runs and returns 0 at sea level" (the kind of test that makes CI green);
* **typical:** one hand-computed value (5 575 m at 50 kPa, from Section 2);
* **strong:** table values, a value below sea level, monotonicity and an array input.

**Exercise 7.1.** Before running the cell: predict which mutants each suite kills.

In [ ]:
H_50KPA_M: Final[float] = (
    5574.43  # Section 2, hand calculation (to 0.01 m with the library)
)


def weak_suite(f) -> bool:
    return float(f(ISA_SEA_LEVEL_PRESSURE_PA)) == 0.0


def typical_suite(f) -> bool:
    return weak_suite(f) and abs(float(f(P_TEST_PA)) - H_50KPA_M) < ALTITUDE_TOL_M


def strong_suite(f) -> bool:
    log = np.array(
        [HIGH_PRESSURE_DAY_PA, ISA_SEA_LEVEL_PRESSURE_PA, P_TEST_PA, TROPOPAUSE_PA]
    )
    h = np.asarray(f(log))
    return bool(
        typical_suite(f)
        and abs(float(f(TROPOPAUSE_PA)) - TROPOPAUSE_M) < ALTITUDE_TOL_M
        and float(f(HIGH_PRESSURE_DAY_PA)) < 0.0
        and h.shape == log.shape
        and np.all(np.diff(h) > 0.0)
    )


scores = {}
for suite in (weak_suite, typical_suite, strong_suite):
    report = runner.run(suite)
    scores[suite.__name__] = report.score
    print(
        f"{suite.__name__:<14} {report}   survived: {', '.join(report.survived) or '-'}"
    )

**Exercise 7.2 (the deleted-tests story in miniature).** An agent asked to "make the tests pass"
for candidate D deletes the assertions D fails (below sea level, array shape, monotonic). Write that weakened suite and measure what
the deletion costs. A mutation score tracked in CI turns a silent deletion into a visible drop.

In [ ]:
def weakened_suite(f) -> bool:
    """strong_suite without the below-sea-level, array-shape and monotonic assertions."""
    return True  # TODO: keep only the typical_suite and tropopause assertions


weakened = runner.run(weakened_suite)
print(f"weakened suite: {weakened}   survived: {', '.join(weakened.survived)}")
print("candidate D passes the weakened suite:", weakened_suite(candidate_d))

## 8. Look before you run: static checks

Two checks need no execution at all, so they are safe on code you have not read yet.

1. **Count what you asked for** (Kim and Yegge, Ch. 7, "count your babies"): an assistant can
   return less than requested, or extra things nobody asked for.
2. **Imports that do not resolve.** Spracklen et al. (USENIX Security 2025) generated 576 000 code
   samples with 16 models and found that the share of recommended packages that do not exist
   averaged at least 5.2 % for commercial and 21.7 % for open-source models, with 205 474 unique
   invented names. An attacker can publish malware under such a name. A name that does not
   resolve *here* may just be uninstalled; check the registry and the publisher before installing.

The text below is a constructed answer to the request "write `pressure_altitude`,
`altitude_to_pressure` and a test". The module name in its first import is invented for this
example. **Do not install it.**

In [ ]:
ANSWER_TEXT: Final[str] = """
import numpy as np
import hallucinated_isa_toolkit as isa

def pressure_altitude(p):
    return isa.troposphere.altitude(p)

def test_pressure_altitude():
    assert pressure_altitude(101325) == 0
"""
audit = audit_deliverables(
    ANSWER_TEXT, ["pressure_altitude", "altitude_to_pressure", "test_pressure_altitude"]
)
print("missing:", sorted(audit.missing))
print("imports that do not resolve here:", sorted(unresolvable_imports(ANSWER_TEXT)))
print(
    f"NumPy {np.__version__}: has trapz {hasattr(np, 'trapz')}, has trapezoid {hasattr(np, 'trapezoid')}"
)

The last line is a different trap: an assistant trained on newer code may call functions your
pinned environment does not have, or the reverse. In the course environment (NumPy 1.26)
`np.trapz` exists and `np.trapezoid` does not; in this container's system Python (NumPy 2.5.3)
it was the other way round. Pin versions and run the tests in the pinned environment.

## 9. What the evidence says (and what it does not)

Every number in this table was read in the cited source on 2026-10-16. Negative change = faster.

| Study | Setting | Result |
|---|---|---|
| Peng, Kalliamvakou, Cihon, Demirer (arXiv:2302.06590, 2023) | RCT, 95 freelancers, one task (HTTP server in JavaScript), GitHub Copilot | 55.8 % less time (95 % CI 21 % to 89 %); success rate +7 points, not significant; code quality not studied |
| METR (arXiv:2507.09089, July 2025) | RCT, 16 experienced open-source developers, 246 real issues in their own repositories, early-2025 tools | **+19 % time** (CI +2 % to +39 %); developers had forecast −24 % and afterwards estimated −20 % |
| METR update (blog, 24 Feb 2026) | Late-2025 follow-up | −18 % (CI −38 % to +9 %) returning developers; −4 % (CI −15 % to +9 %) new ones; METR calls this very weak evidence because of selection effects |
| DORA, *Accelerate State of DevOps* 2024 (Google Cloud blog) | Survey | As AI adoption increased: delivery throughput −1.5 %, delivery stability −7.2 % (estimates); 39 % report little or no trust in AI-generated code |
| Pearce et al., "Asleep at the Keyboard?" (arXiv:2108.09293) | 89 security-relevant scenarios, 1 689 Copilot programs | about 40 % vulnerable |
| Perry, Srivastava, Kumar, Boneh (ACM, DOI 10.1145/3576915.3623157; arXiv:2211.03622) | User study, AI assistant (codex-davinci-002) | participants with the assistant wrote significantly less secure code and were more likely to believe it was secure |
| Veracode, 2025 GenAI Code Security Report (press release, 30 July 2025) | Vendor test: 100+ LLMs, 80 tasks | security flaws in 45 % of test cases (vendor report; peer review not stated) |

**Not used, because no source could be found or checked:** the *Vibe Coder's Handbook*
(Ch. 17) gives "63 % of vibe-coding users are non-developers", "41 % of all code shipped at
major companies is AI-generated", a "$4.7 billion" market and "92 % of US developers use AI
tools daily" without references. Its "170 vulnerable apps out of 1 645" traces to third-party
posts about CVE-2025-48757 (Lovable row-level security), not to the CVE record itself. Treat
such numbers the way you treat an AI answer: unverified until shown otherwise.

In [ ]:
# Change in task completion time with AI (negative = faster), as reported; CI where reported.
evidence = pd.DataFrame(
    [
        ("Peng et al. 2023\n(1 greenfield task)", -55.8, -89.0, -21.0),
        ("METR early 2025\n(own repositories)", 19.0, 2.0, 39.0),
        ("METR late 2025\n(returning devs)", -18.0, -38.0, 9.0),
        ("METR late 2025\n(new devs)", -4.0, -15.0, 9.0),
    ],
    columns=["study", "change_pct", "ci_low", "ci_high"],
)
fig, ax = plt.subplots(figsize=(7.0, 3.2))
y = np.arange(len(evidence))[::-1]
ax.errorbar(
    evidence["change_pct"],
    y,
    xerr=[
        evidence["change_pct"] - evidence["ci_low"],
        evidence["ci_high"] - evidence["change_pct"],
    ],
    fmt="o",
    capsize=4,
    color="#1f3a5f",
)
ax.axvline(0.0, color="grey", lw=1)
ax.set_yticks(y, evidence["study"])
ax.set_xlabel("change in completion time with AI (%)  [negative = faster]")
ax.set_title("Same question, different settings, different answers")
plt.tight_layout()
plt.show()

Read the plot as an engineer reads test data from different rigs: the settings differ (a
greenfield toy task versus large, familiar code bases), the tools changed between 2023 and 2025,
and two of the intervals include zero. The defensible conclusion is narrow: **AI assistance can
save a lot of time on well-specified, self-contained tasks, and can cost time on complex work in
code you know well; developers' own impressions are not a reliable measure.** That is exactly
why the verification habits of Sections 3 to 8 matter.

## 10. A working method: specify, generate small, verify, commit

Paraphrasing the five rules of Kim and Yegge (Ch. 4) for an engineering lab:

1. **Delegate well-defined, small tasks** with clear, checkable success criteria.
2. **Supervise more** when the task is novel, complex or safety-relevant.
3. **Set guardrails:** say what the assistant may and may not change (no test files, no data,
   no hardware commands). In their CNC story an agent proposed wiping a machine's storage
   device; keep agents away from lab hardware and real data.
4. **Check the work often:** run the checks above after every small step, not at the end.
5. **Keep persistent references:** a short project file with units, conventions and pinned
   versions that every prompt starts from.

A prompt template that makes the answer verifiable:

```text
Task: <one function or class>. Inputs: <names, units, ranges>. Output: <name, unit>.
Conventions: <sign, frame, spacing, what counts as invalid input>.
Environment: Python 3.11, NumPy 1.26, pandas 2.1; no other packages.
Reference values I will test against: <two or three, with source>.
First list your assumptions and the edge cases. Then the code. Then pytest tests.
Do not modify existing tests.
```

And the safe-use checklist for this course:

* Never paste data, code or documents your lab or sponsor has not cleared for external services.
* Commit (Git) before letting an agent change files, so every step can be undone.
* Read every line before it runs; run unknown code in a throw-away environment.
* Install nothing an answer imports until you have checked that the package exists and who
  publishes it.
* You sign the result. "The AI wrote it" is not an engineering justification.

In [ ]:
# Notebook self-check (all must pass before you hand in the lab).
assert results.loc["A"].eq("pass").all()  # rounded constants, still within 1 m
assert results.loc["B", "table values"] == "fail"  # universal gas constant
assert results.loc["C", "table values"] == "fail"  # hPa in, feet out
assert results.loc["D", "below sea level"] == "fail"  # clamp hidden by "safety"
assert results.loc["D", "table values"] == "pass"  # ...which the table alone misses
assert abs(h_by_formula - 5574.43) < 0.01
assert abs(observed_order(order_table["rk4_step"], STEPS)[-1] - 4.0) < 0.05
assert observed_order(order_table["rk4_step_typo"], STEPS)[-1] < 2.5
assert (
    pitch_error(rk4_step_typo, SINGLE_RUN_STEP) < SINGLE_RUN_TOL
)  # the one-off test is fooled
assert scores["weak_suite"] < scores["typical_suite"] < scores["strong_suite"] == 1.0
assert weakened.score < 1.0 and weakened_suite(candidate_d)
assert audit.missing == {"altitude_to_pressure"}
print("All bonus-week checks passed.")

## What we now know how to do

* Tell vibe coding (accept without reading) from AI-assisted engineering (specify, review, test,
  explain), and say who is responsible for the result.
* Verify AI-written numerical code with a portfolio of checks: reference values with sources,
  properties, differential tests against a trusted implementation, and the observed order of
  accuracy against an exact solution.
* Recognise the typical defects: wrong constants, unit contracts, "defensive" clamps that change
  the physics, a one-character typo that lowers the order of a method, version drift.
* Measure whether tests can be trusted with mutation testing, and notice when tests are weakened.
* Run static checks first: missing deliverables and imports that do not resolve.
* Read productivity and security studies critically, and refuse unsourced statistics.